# Generar un PowerPoint por cada subcarpeta (imágenes + Markdown)

Este cuaderno recorre una **carpeta raíz**, localiza sus **subcarpetas** y genera **un archivo PowerPoint por cada subcarpeta**.

Cada presentación:
- toma como nombre el de la subcarpeta;
- incluye las **imágenes** y ficheros **`.md`** de esa subcarpeta;
- crea una **diapositiva por cada elemento** ordenado de forma natural;
- inserta las imágenes **completas, sin recorte**, manteniendo la proporción;
- añade **bandas blancas** cuando sea necesario;
- convierte cada `.md` en una **diapositiva de texto centrado**.

En las diapositivas generadas desde `.md`:
- los títulos `#` aparecen en **azul**, tamaño **44** y **negrita**;
- los títulos `##` aparecen en **naranja**, tamaño **28** y **negrita**;
- los títulos `###` aparecen en **verde**, tamaño **22** y **negrita**;
- el resto del contenido del fichero se incluye completo como texto normal.

## Qué hace
- Lee las subcarpetas de una carpeta raíz.
- Para cada subcarpeta, busca imágenes y `.md`.
- Genera un `.pptx` con el nombre de la subcarpeta.
- Guarda todas las presentaciones en una carpeta de salida.
- Exporta cada presentación a PDF al terminar.

## Formatos admitidos
- Imágenes: `.jpg`, `.jpeg`, `.png`, `.bmp`, `.gif`, `.tif`, `.tiff`, `.webp`
- Texto: `.md`



In [5]:
# Instala dependencias si fuera necesario
# %pip install python-pptx pillow
# Requiere LibreOffice (`soffice`) para exportar de .pptx a .pdf


In [6]:
from pathlib import Path
import re
import shutil
import subprocess
from PIL import Image
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN, MSO_VERTICAL_ANCHOR
from pptx.dml.color import RGBColor

# =========================
# CONFIGURACIÓN
# =========================
CARPETA_RAIZ = Path('../img')   # Carpeta que contiene subcarpetas
CARPETA_SALIDA = Path('.')           # Carpeta donde se guardarán los .pptx y .pdf
FORMATO_DIAPOSITIVA = '16:9'                     # Opciones: '16:9' o '4:3'
COLOR_FONDO = 'FFFFFF'                          # Blanco
EXPORTAR_TAMBIEN_A_PDF = True                   # Requiere LibreOffice (`soffice`)

EXTENSIONES_VALIDAS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff', '.webp'}
EXTENSIONES_TEXTO = {'.md'}

COLOR_LINEA_1 = '0270c9'  # azul
COLOR_LINEA_2 = 'f36715'  # naranja
COLOR_LINEA_3 = '298f3d'  # verde
COMANDO_SOFFICE = shutil.which('soffice') or shutil.which('libreoffice')

In [7]:
def orden_natural(texto: str):
    """Ordena cadenas con números de forma natural: img2 antes que img10."""
    return [int(fragmento) if fragmento.isdigit() else fragmento.lower()
            for fragmento in re.split(r'(\d+)', texto)]


def sanitizar_nombre_archivo(nombre: str) -> str:
    """Limpia caracteres problemáticos para usar el nombre como archivo de salida."""
    nombre = re.sub(r'[\/:*?"<>|]+', '_', nombre.strip())
    return nombre or 'presentacion'


def obtener_subcarpetas(carpeta_raiz: Path):
    if not carpeta_raiz.exists():
        raise FileNotFoundError(f'No existe la carpeta raíz: {carpeta_raiz.resolve()}')

    subcarpetas = [p for p in carpeta_raiz.iterdir() if p.is_dir()]
    subcarpetas.sort(key=lambda p: orden_natural(p.name))

    if not subcarpetas:
        raise ValueError(f'No se han encontrado subcarpetas en: {carpeta_raiz.resolve()}')

    return subcarpetas


def obtener_elementos(carpeta: Path):
    elementos = [
        p for p in carpeta.iterdir()
        if p.is_file() and (p.suffix.lower() in EXTENSIONES_VALIDAS or p.suffix.lower() in EXTENSIONES_TEXTO)
    ]
    elementos.sort(key=lambda p: orden_natural(p.name))
    return elementos


def configurar_tamano_presentacion(prs: Presentation, formato: str = '16:9'):
    if formato == '16:9':
        prs.slide_width = Inches(13.333)
        prs.slide_height = Inches(7.5)
    elif formato == '4:3':
        prs.slide_width = Inches(10)
        prs.slide_height = Inches(7.5)
    else:
        raise ValueError("FORMATO_DIAPOSITIVA debe ser '16:9' o '4:3'")


def poner_fondo_solido(slide, color_hex='FFFFFF'):
    fondo = slide.background
    fill = fondo.fill
    fill.solid()
    fill.fore_color.rgb = RGBColor.from_string(color_hex)


def ajustar_imagen_sin_recorte(slide, ruta_imagen: Path, slide_width: int, slide_height: int):
    """
    Inserta la imagen completa dentro de la diapositiva, manteniendo proporciones.
    Puede dejar bandas laterales o superior/inferior.
    """
    with Image.open(ruta_imagen) as img:
        img_width_px, img_height_px = img.size

    escala = min(slide_width / img_width_px, slide_height / img_height_px)
    ancho_final = int(img_width_px * escala)
    alto_final = int(img_height_px * escala)

    izquierda = int((slide_width - ancho_final) / 2)
    arriba = int((slide_height - alto_final) / 2)

    slide.shapes.add_picture(str(ruta_imagen), izquierda, arriba, width=ancho_final, height=alto_final)


def leer_bloques_markdown(ruta_md: Path):
    with open(ruta_md, 'r', encoding='utf-8') as f:
        lineas = [linea.rstrip() for linea in f.readlines()]

    return lineas


def obtener_estilo_markdown(linea: str):
    estilos_titulos = {
        1: {'color': COLOR_LINEA_1, 'size': 44, 'bold': True},
        2: {'color': COLOR_LINEA_2, 'size': 28, 'bold': True},
        3: {'color': COLOR_LINEA_3, 'size': 22, 'bold': True},
    }

    coincidencia = re.match(r'^(#{1,3})\s+(.*)$', linea)
    if coincidencia:
        nivel = len(coincidencia.group(1))
        texto = coincidencia.group(2).strip()
        return texto, estilos_titulos[nivel]

    return linea.strip(), {'color': '000000', 'size': 18, 'bold': False}


def crear_slide_desde_markdown(slide, ruta_md: Path, slide_width: int, slide_height: int):
    lineas = leer_bloques_markdown(ruta_md)

    caja = slide.shapes.add_textbox(0, 0, slide_width, slide_height)
    text_frame = caja.text_frame
    text_frame.clear()
    text_frame.word_wrap = True
    text_frame.vertical_anchor = MSO_VERTICAL_ANCHOR.MIDDLE
    text_frame.margin_left = 0
    text_frame.margin_right = 0
    text_frame.margin_top = 0
    text_frame.margin_bottom = 0

    for i, linea in enumerate(lineas):
        parrafo = text_frame.paragraphs[0] if i == 0 else text_frame.add_paragraph()
        parrafo.alignment = PP_ALIGN.CENTER

        if not linea:
            continue

        texto, estilo = obtener_estilo_markdown(linea)
        if not texto:
            continue

        run = parrafo.add_run()
        run.text = texto
        run.font.name = 'Arial'
        run.font.size = Pt(estilo['size'])
        run.font.bold = estilo['bold']
        run.font.color.rgb = RGBColor.from_string(estilo['color'])

    return len(lineas)


def exportar_presentacion_a_pdf(ruta_pptx: Path, carpeta_salida: Path):
    if not EXPORTAR_TAMBIEN_A_PDF:
        return None, 'Exportación a PDF desactivada por configuración.'

    if not COMANDO_SOFFICE:
        return None, 'No se ha encontrado LibreOffice (`soffice`) en el sistema.'

    carpeta_salida.mkdir(parents=True, exist_ok=True)
    comando = [
        COMANDO_SOFFICE,
        '--headless',
        '--convert-to',
        'pdf',
        '--outdir',
        str(carpeta_salida.resolve()),
        str(ruta_pptx.resolve()),
    ]

    try:
        resultado = subprocess.run(comando, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as exc:
        detalle = exc.stderr.strip() or exc.stdout.strip() or str(exc)
        return None, f'Error exportando a PDF: {detalle}'

    ruta_pdf = carpeta_salida / f'{ruta_pptx.stem}.pdf'
    if not ruta_pdf.exists():
        detalle = resultado.stderr.strip() or resultado.stdout.strip() or 'LibreOffice no generó el archivo esperado.'
        return None, detalle

    return ruta_pdf, None


def crear_presentacion_desde_subcarpeta(subcarpeta: Path, carpeta_salida: Path):
    elementos = obtener_elementos(subcarpeta)
    if not elementos:
        return None

    prs = Presentation()
    configurar_tamano_presentacion(prs, FORMATO_DIAPOSITIVA)

    slide_width = prs.slide_width
    slide_height = prs.slide_height
    layout_en_blanco = prs.slide_layouts[6]

    total_imagenes = 0
    total_markdown = 0

    for elemento in elementos:
        print(elemento)
        slide = prs.slides.add_slide(layout_en_blanco)
        poner_fondo_solido(slide, COLOR_FONDO)

        if elemento.suffix.lower() in EXTENSIONES_VALIDAS:
            ajustar_imagen_sin_recorte(slide, elemento, slide_width, slide_height)
            total_imagenes += 1
        elif elemento.suffix.lower() in EXTENSIONES_TEXTO:
            crear_slide_desde_markdown(slide, elemento, slide_width, slide_height)
            total_markdown += 1

    carpeta_salida.mkdir(parents=True, exist_ok=True)
    nombre_ppt = sanitizar_nombre_archivo(subcarpeta.name) + '.pptx'
    ruta_salida = carpeta_salida / nombre_ppt
    prs.save(ruta_salida)
    ruta_pdf, error_pdf = exportar_presentacion_a_pdf(ruta_salida, carpeta_salida)

    return {
        'subcarpeta': subcarpeta,
        'ruta_salida': ruta_salida,
        'ruta_pdf': ruta_pdf,
        'error_pdf': error_pdf,
        'total_elementos': len(elementos),
        'total_imagenes': total_imagenes,
        'total_markdown': total_markdown,
    }



In [8]:
# Crear una presentación por cada subcarpeta
subcarpetas = obtener_subcarpetas(CARPETA_RAIZ)

print('Subcarpetas encontradas:')
for subcarpeta in subcarpetas:
    print(f' - {subcarpeta.name}')

resultados = []
subcarpetas_sin_elementos = []

for subcarpeta in subcarpetas:
    resultado = crear_presentacion_desde_subcarpeta(subcarpeta, CARPETA_SALIDA)
    if resultado is None:
        subcarpetas_sin_elementos.append(subcarpeta.name)
    else:
        resultados.append(resultado)

print('Resumen de presentaciones generadas:')
if resultados:
    for r in resultados:
        print(f"- {r['subcarpeta'].name} -> {r['ruta_salida'].resolve()}")
        print(f"    elementos: {r['total_elementos']} | imágenes: {r['total_imagenes']} | md: {r['total_markdown']}")
        if r['ruta_pdf'] is not None:
            print(f"    pdf: {r['ruta_pdf'].resolve()}")
        elif r['error_pdf']:
            print(f"    aviso pdf: {r['error_pdf']}")
else:
    print('No se ha generado ninguna presentación.')

if subcarpetas_sin_elementos:
    print('Subcarpetas sin imágenes ni Markdown válidos:')
    for nombre in subcarpetas_sin_elementos:
        print(f' - {nombre}')



Subcarpetas encontradas:
 - 000-curso-practico-IA-generativa
 - 010-introduccion-IA-generativa
 - 020-introduccion-LLMs
 - 040-prompt-enginering
 - 050-comparativa-modelos-llm
 - 060-herramientas-IA-gratuitas
../img/000-curso-practico-IA-generativa/000-curso-practico-IA-generativa.md
../img/000-curso-practico-IA-generativa/010-010-ia-generativa.md
../img/000-curso-practico-IA-generativa/010-020-ia-generativa-que.png
../img/000-curso-practico-IA-generativa/010-030-sectores-y-casos-de-uso.png
../img/000-curso-practico-IA-generativa/010-040-beneficios-y-desafios.png
../img/000-curso-practico-IA-generativa/010-050-limitaciones-y-riesgos-de-seguridad.png
../img/000-curso-practico-IA-generativa/010-060-retos-y-competencias.png
../img/000-curso-practico-IA-generativa/010-070-tendencias-tecnologicas-y-regulacion.png
../img/000-curso-practico-IA-generativa/010-075-historia-ia-generativa.md
../img/000-curso-practico-IA-generativa/010-080-origenes.png
../img/000-curso-practico-IA-generativa/010-0

## Uso

1. Crea una carpeta raíz, por ejemplo `carpetas_presentaciones`.
2. Dentro de ella, crea una subcarpeta por cada presentación.
3. En cada subcarpeta, guarda las imágenes numeradas y, si quieres, también ficheros `.md`.
4. Ejecuta todas las celdas.
5. Se generará un archivo `.pptx` y, si `LibreOffice` está disponible, también un `.pdf` por cada subcarpeta dentro de la carpeta de salida.

## Ejemplo de estructura

```text
carpetas_presentaciones/
├── Tema 1/
│   ├── 1.md
│   ├── 2.png
│   └── 3.jpg
├── Tema 2/
│   ├── 1.jpg
│   ├── 2.md
│   └── 3.png
└── Unidad 3/
    ├── 01.md
    ├── 02.webp
    └── 10.png
```

## Formato esperado de cada Markdown
- `# Título`: azul, tamaño 44 y negrita.
- `## Subtítulo`: naranja, tamaño 28 y negrita.
- `### Apartado`: verde, tamaño 22 y negrita.
- El resto de líneas se añade como texto normal.

Se procesa todo el fichero, no solo las primeras líneas.

## Nota
Los elementos de cada subcarpeta se ordenan conjuntamente por nombre. Así, por ejemplo, `1.jpg`, `2.md` y `3.png` aparecerán en ese orden dentro de su presentación.

